In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [3]:
df.shape

(2000, 8)

In [4]:
df.head(3)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [5]:
df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [6]:
df=df.drop('id',axis=1)

In [7]:
answer=df['answer']
df=df.drop('answer',axis=1)

In [8]:
answer.value_counts(normalize=True)

answer
B    0.2450
C    0.2295
A    0.1845
D    0.1790
E    0.1620
Name: proportion, dtype: float64

In [9]:
df.shape

(2000, 6)

In [10]:
df['prompt_len']=df['prompt'].str.len()
for col in ['A','B','C','D','E']:
    df[f'{col}_len']=df[col].str.len()

def get_overlaps(row):
    prompt_words=set(row['prompt'].split())
    overlaps={}
    for col in ['A','B','C','D','E']:
        option_words=set(row[col].split())
        overlaps[f'{col}_overlap']=len(prompt_words & option_words)
    return pd.Series(overlaps)

overlaps_df=df.apply(get_overlaps,axis=1)
df=pd.concat([df,overlaps_df],axis=1)

In [11]:
from sklearn.model_selection import train_test_split

x_train,x_val,y_train,y_val=train_test_split(df,answer,train_size=0.8,random_state=1)

In [12]:
def score_to_labels(scores,top1=False):
    labels=[]
    top1_labels=[]
    for score in scores:
        scored=np.argsort(score)[::-1][:3]
        top_3=[chr(65+i) for i in scored]
        labels.append(" ".join(top_3))
        top1_labels.append(chr(65+np.argsort(score)[::-1][0]))
    if not top1:
        return labels
    else:
        return top1_labels

In [13]:
def map_at_3(predictions,ground_truth):
    ap_scores=[]
    for pred, truth in zip(predictions, ground_truth):
        pred_list = pred.split()
        if truth in pred_list:
            ap_scores.append(1.0 / (pred_list.index(truth) + 1))
        else:
            ap_scores.append(0.0)
    return sum(ap_scores) / len(ap_scores)

In [14]:
from sklearn.metrics import accuracy_score,f1_score

def support_metrics(train_scores,val_scores):
    train_acc=accuracy_score(score_to_labels(train_scores,top1=True),y_train)
    val_acc=accuracy_score(score_to_labels(val_scores,top1=True),y_val)
    train_f1=f1_score(score_to_labels(train_scores,top1=True),y_train,average='macro')
    val_f1=f1_score(score_to_labels(val_scores,top1=True),y_val,average='macro')
    return train_acc,val_acc,train_f1,val_f1

In [15]:
!pip install wandb -q

In [16]:
import wandb
from kaggle_secrets import UserSecretsClient
import os

secrets=UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
wandb.login()
wandb_mode=True

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [17]:
def len_scores(row):
    lengths=[row['A_len'],row['B_len'],row['C_len'],row['D_len'],row['E_len']]
    max_len=max(lengths)
    min_len=min(lengths)
    scores=[]
    for length in lengths:
        score=0
        if length==max_len:
            score=0.5
        elif length==min_len:
            score=0.2
        else:
            score=0.3
        scores.append(score)
    return scores

length_train_scores=x_train.apply(len_scores,axis=1).to_list()
length_labels=score_to_labels(length_train_scores)
length_map3=map_at_3(length_labels,y_train)
print(length_map3)

0.5320833333333334


In [18]:
length_val_scores=x_val.apply(len_scores,axis=1).to_list()
length_val_labels=score_to_labels(length_val_scores)
length_val_map3=map_at_3(length_val_labels,y_val)
length_val_map3

0.49666666666666665

In [19]:
from sklearn.metrics import accuracy_score,f1_score

len_train_acc,len_val_acc,len_train_f1,len_val_f1=support_metrics(length_train_scores,length_val_scores)

In [20]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="largestlen-baseline-metrics",id="largelen_v1",resume="allow")
    wandb.log({
        "map_at_3_train":length_map3,   
        "map_at_3_val":length_val_map3,
        "accuracy_at_train":len_train_acc,
        "accuracy_at_val":len_val_acc,
        "f1_at_train":len_train_f1,
        "f1_at_val":len_val_f1,
        "metric":"Longest-Length"
        })
    wandb.finish()

wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260723_030148-largelen_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run largestlen-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/largelen_v1
wandb: updating run metadata
wandb: uploading history steps 41-41, summary
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.405
wandb:   accuracy_at_val 0.3825
wandb:       f1_at_train 0.40268
wandb:         f1_at_val 0.37871
wandb:    map_at_3_train 0.53208
wandb:      map_at_3_val 0.49667
wandb:            metric Longest-Length
wandb: 


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf=TfidfVectorizer(sublinear_tf=True,ngram_range=(1,2))
all_text=x_train['prompt'].tolist()+x_train['A'].tolist()+x_train['B'].tolist()+x_train['C'].tolist()+x_train['D'].tolist()+x_train['E'].tolist()
tfidf.fit(all_text)

TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)

In [22]:
tfidf_train_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_train['prompt']+" "+x_train[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_train_scores.append(np.asarray(scores).flatten())
tfidf_train_scores=np.array(tfidf_train_scores).T

In [23]:
tfidf_val_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(x_val['prompt']+" "+x_val[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_val_scores.append(np.asarray(scores).flatten())
tfidf_val_scores=np.array(tfidf_val_scores).T

In [24]:
tfidf_train_labels=score_to_labels(tfidf_train_scores,top1=False)
tfidf_train_map3=map_at_3(tfidf_train_labels,y_train)
print(tfidf_train_map3)

tfidf_val_labels=score_to_labels(tfidf_val_scores,top1=False)
tfidf_val_map3=map_at_3(tfidf_val_labels,y_val)
print(tfidf_val_map3)

0.5355208333333333
0.5320833333333334


In [25]:
tfidf_train_acc,tfidf_val_acc,tfidf_train_f1,tfidf_val_f1=support_metrics(tfidf_train_scores,tfidf_val_scores)

In [26]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="tfidf-baseline-metrics",id="tfidf_v1",resume="allow")
    wandb.log({
        "map_at_3_train":tfidf_train_map3,   
        "map_at_3_val":tfidf_val_map3,
        "accuracy_at_train":tfidf_train_acc,
        "accuracy_at_val":tfidf_val_acc,
        "f1_at_train":tfidf_train_f1,
        "f1_at_val":tfidf_val_f1,
        "vectorizer":"TfidfVectorizer",
        "sublinear_tf":True})
    wandb.finish()

wandb: setting up run tfidf_v1
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260723_030205-tfidf_v1
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run tfidf-baseline-metrics
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/tfidf_v1
wandb: uploading summary; updating run metadata
wandb: uploading summary
wandb: uploading history steps 94-94, summary
wandb: 
wandb: Run history:
wandb: accuracy_at_train ▁
wandb:   accuracy_at_val ▁
wandb:       f1_at_train ▁
wandb:         f1_at_val ▁
wandb:    map_at_3_train ▁
wandb:      map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb: accuracy_at_train 0.34813
wandb:   accuracy_at_val 0.3525
wandb:       f1_at_train 0.34639
wandb:         f1_at_val 0.35202
wandb:    map_at_3_train 0.53552
wandb:      map

In [27]:
label_map={"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

y_train_idx=y_train.map(label_map).values if y_train.dtype=='object' else y_train.values
y_val_idx=y_val.map(label_map).values if y_val.dtype=='object' else y_val.values

In [28]:
import torch
import numpy as np
import random

seed=42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [29]:
from gensim.models import Word2Vec
np.random.seed(1)
from numpy.random import seed
seed(1)

sentences=[text.split() for text in all_text]
word2vec=Word2Vec(sentences,vector_size=10,window=1,min_count=1,seed=1,workers=1,epochs=10)

In [30]:
option_cols=['A','B','C','D','E']
def sentence_embedding(text):
    words=text.split()
    embedding=[word2vec.wv[word] for word in words if word in word2vec.wv]
    if len(embedding)==0:
        return np.zeros(word2vec.vector_size)
    return np.mean(embedding,axis=0)

In [31]:
prompts=np.vstack(x_train['prompt'].apply(sentence_embedding))
w2v_train_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_train[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(prompts,option_matrix))
    w2v_train_scores.append(scores)
w2v_train_scores=np.array(w2v_train_scores).T

In [32]:
val_prompts=np.vstack(x_val['prompt'].apply(sentence_embedding))
w2v_val_scores=[]

for col in option_cols:
    option_matrix=np.vstack(x_val[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(val_prompts,option_matrix))
    w2v_val_scores.append(scores)
w2v_val_scores=np.array(w2v_val_scores).T

In [33]:
w2v_train_labels=score_to_labels(w2v_train_scores)
w2v_train_map3=map_at_3(w2v_train_labels,y_train)
print(w2v_train_map3)

w2v_val_labels=score_to_labels(w2v_val_scores)
w2v_val_map3=map_at_3(w2v_val_labels,y_val)
print(w2v_val_map3)

0.3885416666666666
0.37416666666666665


In [34]:
from torch.utils.data import TensorDataset,DataLoader
from sklearn.preprocessing import StandardScaler

ss=StandardScaler()

x_train_t=torch.tensor(ss.fit_transform(np.concatenate((tfidf_train_scores,np.array(length_train_scores),w2v_train_scores,x_train[['A_overlap','B_overlap','C_overlap','D_overlap','E_overlap']].values),axis=1)),dtype=torch.float32)
x_val_t=torch.tensor(ss.transform(np.concatenate((tfidf_val_scores,np.array(length_val_scores),w2v_val_scores,x_val[['A_overlap','B_overlap','C_overlap','D_overlap','E_overlap']].values),axis=1)),dtype=torch.float32)
y_train_t=torch.tensor(y_train_idx,dtype=torch.long)
y_val_t=torch.tensor(y_val_idx,dtype=torch.long)

In [35]:
from torch.utils.data import TensorDataset,DataLoader

train_ds=TensorDataset(x_train_t,y_train_t)
train_loader=DataLoader(train_ds,batch_size=32,shuffle=True)

In [36]:
import torch.nn as nn

class MCQANN(nn.Module):
    def __init__(self,input_dim=20,num_choices=5,hidden_dim=64):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim,hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim,num_choices)
        )
    def forward(self,x):
        return self.net(x)

model=MCQANN(input_dim=20)

In [37]:
wandb.init(project="24f2002314-t22026",name="MCQ-ANN-scaled",id="ann_version_scaled2",resume="allow")
wandb.log({
        "lr":2e-3,   
        "weight_decay":1e-4,
        "epochs": 400,
        "patience": 15,
         "Batch Normalisation":True,
        "hidden_dimension":64,
        "batch_size": 32,
        "architecture": "MCQANN",
        "input_dim": x_train_t.shape[1],})

wandb: setting up run ann_version_scaled2
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260723_030252-ann_version_scaled2
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run MCQ-ANN-scaled
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/ann_version_scaled2


In [38]:
import copy

criteria=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)
device="cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

best_val_loss=float('inf')
patience=15
patience_count=0
best_val_map3=0
EPOCHS=400
for epoch in range(EPOCHS):
    model.train()
    total_loss=0
    for x,y in train_loader:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        output=model(x)
        loss=criteria(output,y)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()

    if epoch%1==0:
        model.eval()
        with torch.no_grad():
            val_ds=x_val_t.to(device)
            val_out=model(val_ds)
            val_loss=criteria(val_out,y_val_t.to(device))
            val_acc=(val_out.argmax(dim=1).cpu()==y_val_t).float().mean()
            val_scores=val_out.cpu().numpy()
            val_labels=score_to_labels(val_scores,top1=False)
            val_map3=map_at_3(val_labels, y_val)
            avg_train_loss=total_loss/len(train_loader)
        print(f'Epoch :{epoch} - train loss : {avg_train_loss:.4f} val loss : {val_loss:.4f} val_acc={val_acc:.4f} val_map3={val_map3:.4f}')
        wandb.log({
            "epoch": epoch,
            "train_loss": avg_train_loss,
            "val_loss": val_loss.item(),
            "val_acc": val_acc.item(),
            "val_map3": val_map3
        })
            
    if val_map3>=best_val_map3:
        best_val_loss=val_loss
        best_val_map3=val_map3
        patience_count=0
        best_state=copy.deepcopy(model.state_dict())
    else:
        patience_count+=1
        if patience_count>=patience:
            print(f"Early stopping at epoch {epoch}")
            break
model.load_state_dict(best_state)

Epoch :0 - train loss : 1.5463 val loss : 1.4903 val_acc=0.3975 val_map3=0.5629
Epoch :1 - train loss : 1.3945 val loss : 1.3458 val_acc=0.4775 val_map3=0.6342
Epoch :2 - train loss : 1.2565 val loss : 1.2476 val_acc=0.5100 val_map3=0.6687
Epoch :3 - train loss : 1.1712 val loss : 1.1648 val_acc=0.5625 val_map3=0.7075
Epoch :4 - train loss : 1.1007 val loss : 1.1013 val_acc=0.6075 val_map3=0.7358
Epoch :5 - train loss : 1.0470 val loss : 1.0388 val_acc=0.6500 val_map3=0.7575
Epoch :6 - train loss : 0.9812 val loss : 0.9945 val_acc=0.6600 val_map3=0.7692
Epoch :7 - train loss : 0.9359 val loss : 0.9172 val_acc=0.6850 val_map3=0.7913
Epoch :8 - train loss : 0.8901 val loss : 0.8727 val_acc=0.7100 val_map3=0.8075
Epoch :9 - train loss : 0.8411 val loss : 0.8282 val_acc=0.7125 val_map3=0.8142
Epoch :10 - train loss : 0.8082 val loss : 0.7871 val_acc=0.7375 val_map3=0.8321
Epoch :11 - train loss : 0.7687 val loss : 0.7461 val_acc=0.7575 val_map3=0.8446
Epoch :12 - train loss : 0.7288 val lo

<All keys matched successfully>

In [39]:
wandb.run.summary["best_val_loss"]=best_val_loss.item()
wandb.run.summary["best_val_map3"]=best_val_map3   
wandb.run.summary["stopped_epoch"]=epoch
torch.save(best_state,"best_model.pt")
artifact=wandb.Artifact("mcq_ann_best_model",type="model")
artifact.add_file("best_model.pt")
wandb.log_artifact(artifact)
wandb.finish()

wandb: uploading history steps 352-405, summary, console lines 0-52; uploading artifact mcq_ann_best_model; updating run metadata
wandb: uploading history steps 352-405, summary, console lines 0-52
wandb: uploading history steps 406-491, summary, console lines 53-139
wandb: 
wandb: Run history:
wandb:       batch_size ▁
wandb:            epoch ▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
wandb:           epochs ▁
wandb: hidden_dimension ▁
wandb:        input_dim ▁
wandb:               lr ▁
wandb:         patience ▁
wandb:       train_loss █▇▆▅▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:          val_acc ▁▄▄▅▅▆▆▇▇▇▇▇███▇████████████████████████
wandb:         val_loss █▇▆▅▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:               +2 ...
wandb: 
wandb: Run summary:
wandb: Batch Normalisation True
wandb:        architecture MCQANN
wandb:          batch_size 32
wandb:       best_val_loss 0.09495
wandb:       best_val_map3 0.9875
wandb:               epoch 138
wandb:              epochs 400
wandb

In [40]:
from datasets import Dataset

label_map={'A':0,'B':1,'C':2,'D':3,'E':4}

x_tr=x_train.copy()
x_v=x_val.copy()
x_tr['label']=y_train.map(label_map)
x_v['label']=y_val.map(label_map)

train_dataset=Dataset.from_pandas(x_tr.reset_index(drop=True))
val_dataset=Dataset.from_pandas(x_v.reset_index(drop=True))

In [41]:
from transformers import AutoModelForMultipleChoice,AutoTokenizer,TrainingArguments,Trainer
import torch

model_name1='google/electra-base-discriminator'
tokeniser1=AutoTokenizer.from_pretrained(model_name1)
model1=AutoModelForMultipleChoice.from_pretrained(model_name1,ignore_mismatched_sizes=True)

model_name2='roberta-base'
tokeniser2=AutoTokenizer.from_pretrained(model_name2)
model2=AutoModelForMultipleChoice.from_pretrained(model_name2,ignore_mismatched_sizes=True)

def preprocessing(data,tokeniser):
    prompts=[[p]*5 for p in data['prompt']]
    options=[[data[o][i] for o in ['A','B','C','D','E']] for i in range(len(data['prompt']))]
    prompts=sum(prompts,[])
    options=sum(options,[])
    processed=tokeniser(prompts,options,truncation=True,max_length=256,padding='max_length')
    result={k:[v[i:i+5] for i in range(0,len(v),5)] for k,v in processed.items()}
    if "label" in data:
         result["label"]=data["label"]

    return result

processed_train1=train_dataset.map(lambda x:preprocessing(x,tokeniser1),batched=True,remove_columns=train_dataset.column_names)
processed_val1=val_dataset.map(lambda x:preprocessing(x,tokeniser1),batched=True,remove_columns=val_dataset.column_names)
processed_train2=train_dataset.map(lambda x:preprocessing(x,tokeniser2),batched=True,remove_columns=train_dataset.column_names)
processed_val2=val_dataset.map(lambda x:preprocessing(x,tokeniser2),batched=True,remove_columns=val_dataset.column_names)

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

... (10 outputs left out)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [42]:
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits,labels=eval_pred
    top3=np.argsort(-logits,axis=1)[:,:3]
    ap=[1.0/(list(row).index(l)+1) if l in row else 0.0 for row,l in zip(top3,labels)]
    acc=(np.argmax(logits, axis=1)==labels).mean()
    preds=np.argmax(logits,axis=1)
    f1_macro=f1_score(labels,preds,average="macro")
    return {"map_at_3":np.mean(ap),"accuracy":acc,"f1_macro":f1_macro}

In [43]:
"""del model
torch.cuda.empty_cache()"""

'del model\ntorch.cuda.empty_cache()'

In [44]:
processed_train1.set_format(type="torch",columns=["input_ids","attention_mask","label"])
processed_val1.set_format(type="torch",columns=["input_ids","attention_mask","label"])

processed_train2.set_format(type="torch",columns=["input_ids","attention_mask","label"])
processed_val2.set_format(type="torch",columns=["input_ids","attention_mask","label"])

train_args1=TrainingArguments(output_dir="./electra-base-mcq",learning_rate=4e-5,eval_strategy="epoch",gradient_checkpointing=False,save_strategy="epoch",num_train_epochs=6,warmup_steps=500,remove_unused_columns=False,max_grad_norm=1.0,
                             fp16=False,weight_decay=0.01,logging_steps=10,metric_for_best_model="map_at_3",per_device_train_batch_size=8,per_device_eval_batch_size=8,load_best_model_at_end=True)

trainer1=Trainer(model=model1,args=train_args1,train_dataset=processed_train1,eval_dataset=processed_val1,compute_metrics=compute_metrics)
trainer1.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Map At 3,Accuracy,F1 Macro
1,3.179156,3.141469,0.697500,0.547500,0.543191
2,1.834105,1.638906,0.863750,0.787500,0.789032
3,0.847586,0.664047,0.959167,0.937500,0.936014
4,0.312554,0.202965,0.998750,0.997500,0.997334
5,0.158914,0.042589,1.000000,1.000000,1.000000
6,0.004581,0.000287,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (5 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=600, training_loss=1.3037468414132793, metrics={'train_runtime': 1151.0816, 'train_samples_per_second': 8.34, 'train_steps_per_second': 0.521, 'total_flos': 6314608631808000.0, 'train_loss': 1.3037468414132793, 'epoch': 6.0})

In [45]:
train_args2=TrainingArguments(output_dir="./roberta-base-mcq",learning_rate=1e-5,eval_strategy="epoch",gradient_checkpointing=False,save_strategy="epoch",num_train_epochs=6,warmup_steps=500,remove_unused_columns=False,max_grad_norm=1.0,
                             fp16=False,weight_decay=0.01,logging_steps=10,metric_for_best_model="map_at_3",per_device_train_batch_size=8,per_device_eval_batch_size=8,load_best_model_at_end=True)

trainer2=Trainer(model=model2,args=train_args2,train_dataset=processed_train2,eval_dataset=processed_val2,compute_metrics=compute_metrics)
trainer2.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Map At 3,Accuracy,F1 Macro
1,3.226321,3.217727,0.440417,0.270000,0.264020
2,3.237207,3.202968,0.587917,0.422500,0.419583
3,2.965882,2.870942,0.692500,0.515000,0.508045
4,2.162151,1.807989,0.832083,0.740000,0.735185
5,1.323313,1.103622,0.901667,0.855000,0.855606
6,1.136961,0.755831,0.940417,0.915000,0.914675


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (5 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=600, training_loss=2.493628115653992, metrics={'train_runtime': 1153.3279, 'train_samples_per_second': 8.324, 'train_steps_per_second': 0.52, 'total_flos': 6314608631808000.0, 'train_loss': 2.493628115653992, 'epoch': 6.0})

In [46]:
model.eval()
with torch.no_grad():
    val_logits=model(x_val_t.to(device)).cpu().numpy()

In [47]:
"""from scipy.special import softmax

val_results1=trainer1.predict(processed_val1)
val_results2=trainer2.predict(processed_val2)

val_logits1=val_results1.predictions
val_logits2=val_results2.predictions
val_logits3=val_logits

val_logits1=softmax(val_logits1,axis=1)
val_logits2=softmax(val_logits2,axis=1)
val_logits3=softmax(val_logits3,axis=1)

best_score1=0
best_score2=0 
best_w=(0.34, 0.33, 0.33)

label_inverse={0:'A', 1:'B', 2:'C', 3:'D', 4:'E'}
val_labels=[label_inverse[int(l)] for l in list(processed_val1['label'])]
val_labels_idx=[ord(l)-65 for l in val_labels]

for w1 in np.arange(0.0,1.01, 0.05):
    for w2 in np.arange(0.0,1.01 - w1,0.05):  
        w3=1.0-w1-w2                     
        if w3<0:
            continue
        val_blend=w1*val_logits1+w2*val_logits2+w3*val_logits3
        val_blend_labels=score_to_labels(val_blend)
        score1=map_at_3(val_blend_labels,val_labels)
        score2=accuracy_score(np.argmax(val_blend,axis=1),val_labels_idx)

        if score1>best_score1:
            best_score1=score1
            best_score2=score2
            best_w=(w1,w2,w3)

print(best_w)
print(best_score1)
print(best_score2)"""

"from scipy.special import softmax\n\nval_results1=trainer1.predict(processed_val1)\nval_results2=trainer2.predict(processed_val2)\n\nval_logits1=val_results1.predictions\nval_logits2=val_results2.predictions\nval_logits3=val_logits\n\nval_logits1=softmax(val_logits1,axis=1)\nval_logits2=softmax(val_logits2,axis=1)\nval_logits3=softmax(val_logits3,axis=1)\n\nbest_score1=0\nbest_score2=0 \nbest_w=(0.34, 0.33, 0.33)\n\nlabel_inverse={0:'A', 1:'B', 2:'C', 3:'D', 4:'E'}\nval_labels=[label_inverse[int(l)] for l in list(processed_val1['label'])]\nval_labels_idx=[ord(l)-65 for l in val_labels]\n\nfor w1 in np.arange(0.0,1.01, 0.05):\n    for w2 in np.arange(0.0,1.01 - w1,0.05):  \n        w3=1.0-w1-w2                     \n        if w3<0:\n            continue\n        val_blend=w1*val_logits1+w2*val_logits2+w3*val_logits3\n        val_blend_labels=score_to_labels(val_blend)\n        score1=map_at_3(val_blend_labels,val_labels)\n        score2=accuracy_score(np.argmax(val_blend,axis=1),val_l

In [48]:
from scipy.special import softmax

val_results1=trainer1.predict(processed_val1)
val_results2=trainer2.predict(processed_val2)

val_logits1=val_results1.predictions
val_logits2=val_results2.predictions

val_logits1=softmax(val_logits1,axis=1)
val_logits2=softmax(val_logits2,axis=1)

best_score1=0
best_score2=0
best_w=0.5

label_inverse={0:'A', 1:'B', 2:'C', 3:'D', 4:'E'}
val_labels=[label_inverse[int(l)] for l in list(processed_val1['label'])]
val_labels_idx=[ord(l)-65 for l in val_labels]

for w in np.arange(0.1,1.0,0.1):
    val_blend=w*val_logits1 +(1-w)*val_logits2
    val_blend_labels=score_to_labels(val_blend)
    score1=map_at_3(val_blend_labels,val_labels)
    score2=accuracy_score(np.argmax(val_blend,axis=1),val_labels_idx)
    print(f"w={w:.1f}, MAP@3={score1:.4f}")
    if score1>best_score1:
        best_score1=score1
        best_score2=score2
        best_w=w

print(best_w)
print(best_score1)
print(best_score2)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


w=0.1, MAP@3=0.9962
w=0.2, MAP@3=0.9975
w=0.3, MAP@3=1.0000
w=0.4, MAP@3=1.0000
w=0.5, MAP@3=1.0000
w=0.6, MAP@3=1.0000
w=0.7, MAP@3=1.0000
w=0.8, MAP@3=1.0000
w=0.9, MAP@3=1.0000
0.30000000000000004
1.0
1.0


In [49]:
if wandb_mode==True:
    wandb.init(project="24f2002314-t22026",name="roberta-electra-ensemble-v2",id="roberta-electra-ensemble2",resume="allow")
    wandb.log({   
        "map_at_3_val":best_score1,
        "accuracy_at_val":best_score2,
        "best_weight":best_w,
        "MODEL":"ROBERTA-ELECTRA-ENSEMBLE",
        "epochs":6,
        "learning_rate":"4e-5 and 1e-5"
         })
    wandb.finish()

wandb: setting up run roberta-electra-ensemble2
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260723_034310-roberta-electra-ensemble2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run roberta-electra-ensemble-v2
wandb: ⭐️ View project at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026
wandb: 🚀 View run at https://wandb.ai/24f2002314-indian-institute-of-technology-madras/24f2002314-t22026/runs/roberta-electra-ensemble2
wandb: updating run metadata; uploading summary
wandb: uploading summary
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: accuracy_at_val ▁
wandb:     best_weight ▁
wandb:          epochs ▁
wandb:    map_at_3_val ▁
wandb: 
wandb: Run summary:
wandb:           MODEL ROBERTA-ELECTRA-ENSE...
wandb: accuracy_at_val 1
wandb:     best_weight 0.3
wandb:          epochs 6
wandb:   learning_rate 4e-5 and 1e-5
wandb:    map_at_3_val 1
wa

In [50]:
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
test['prompt_len']=test['prompt'].str.len()
for col in ['A','B','C','D','E']:
    test[f'{col}_len']=test[col].str.len()

def get_overlaps(row):
    prompt_words=set(row['prompt'].split())
    overlaps={}
    for col in ['A','B','C','D','E']:
        option_words=set(row[col].split())
        overlaps[f'{col}_overlap']=len(prompt_words & option_words)
    return pd.Series(overlaps)

overlaps_test=test.apply(get_overlaps,axis=1)
test=pd.concat([test,overlaps_test],axis=1)

In [51]:
tfidf_test_scores=[]
for col in ['A', 'B', 'C', 'D', 'E']:
    combined=(test['prompt']+" "+test[col]).to_list()
    scores=tfidf.transform(combined).sum(axis=1)
    tfidf_test_scores.append(np.asarray(scores).flatten())
tfidf_test_scores=np.array(tfidf_test_scores).T

test_length_scores=test.apply(len_scores,axis=1).to_list()

In [52]:
test_prompts=np.vstack(test['prompt'].apply(sentence_embedding))
w2v_test_scores=[]

for col in option_cols:
    option_matrix=np.vstack(test[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(test_prompts,option_matrix))
    w2v_test_scores.append(scores)
w2v_test_scores=np.array(w2v_test_scores).T

In [53]:
x_test=np.concatenate((tfidf_test_scores,np.array(test_length_scores),w2v_test_scores,test[['A_overlap','B_overlap','C_overlap','D_overlap','E_overlap']].values), axis=1)
x_test_t=torch.tensor(ss.transform(x_test),dtype=torch.float32)

In [54]:
model.eval()
with torch.no_grad():
    test_logits=model(x_test_t.to(device)).cpu().numpy()

In [55]:
test_dataset=Dataset.from_pandas(test)
processed_test1=test_dataset.map(lambda x:preprocessing(x,tokeniser1),batched=True,remove_columns=test_dataset.column_names)
processed_test2=test_dataset.map(lambda x:preprocessing(x,tokeniser2),batched=True,remove_columns=test_dataset.column_names)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [56]:
test_results1=trainer1.predict(processed_test1)
test_results2=trainer2.predict(processed_test2)
logits1=test_results1.predictions
logits2=test_results2.predictions

logits1=softmax(logits1,axis=1)
logits2=softmax(logits2,axis=1)
test_logits=softmax(test_logits,axis=1)

logits=best_w*logits1+(1-best_w)*logits2
letters=np.array(["A","B","C","D","E"])
top3=np.argsort(logits,axis=1)[:, ::-1][:, :3]
top3=[" ".join(letters[row]) for row in top3]
print(top3[:10])

['A D B', 'B A E', 'B E D', 'E C A', 'C B A', 'D A B', 'E C A', 'B E C', 'C D E', 'B A D']


In [57]:
submission=pd.DataFrame({'id':test['id'],'Prediction':top3})
submission.to_csv('submission.csv',index=False)
print("Model submission saved!")
print(submission.head())

Model submission saved!
   id Prediction
0   1      A D B
1   2      B A E
2   3      B E D
3   4      E C A
4   5      C B A
